# Runway Screening — Production Training v2 (Colab A100)

Adds three transfer-learning sources on top of RDD2022 + verified drone labels:
**UAV-PDD2023** (Zenodo, VOC, top-down 30 m), **HighRPD** (Mendeley, YOLO, top-down 50 m),
and **FOD-A** (Google Drive, VOC, all→fod). Trains YOLOv8m @ 960 as `runway_m_v2`.

**Before you run:** `Runtime → Change runtime type → A100 GPU`, then run top-to-bottom.
Downloads total ~17 GB (13 GB RDD + 2 + 1.6 + 0.4); training is a few hours.

⚠️ **Stop at the VERIFY cell (7) and read its output** before launching the long merge+train —
it confirms each dataset's on-disk format matches what `merge_datasets.py` expects. In particular
HighRPD must be YOLO `.txt`; if it isn't, or its class order differs from `{0,1→crack, 2→pothole}`,
ping back before training (the loader would mislabel silently).

## 1. Confirm the A100

In [ ]:
!nvidia-smi

## 2. Install deps

In [ ]:
!pip -q install ultralytics gdown
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))

## 3. Clone repo (branch `local/nano-bootstrap`)

In [ ]:
!git clone --branch local/nano-bootstrap --single-branch https://github.com/shabazahmadme1-eng/runway_screening.git
%cd runway_screening

## 4. Shared extract helper (handles nested zips)

In [ ]:
import zipfile, os
from pathlib import Path

def extract(zip_path, dest):
    zip_path, dest = Path(zip_path), Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    print('extracting', zip_path.name, '->', dest, flush=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    for nz in sorted(dest.rglob('*.zip')):
        out = nz.parent / nz.stem
        print('  nested', nz.name, flush=True)
        with zipfile.ZipFile(nz) as z:
            z.extractall(out)
        nz.unlink()
    print('  done:', sum(1 for _ in dest.rglob('*.xml')), 'xml |',
          sum(1 for _ in dest.rglob('*.txt')), 'txt |',
          sum(1 for _ in dest.rglob('*.jpg')) + sum(1 for _ in dest.rglob('*.png')), 'img', flush=True)

## 5. Download everything
RDD2022 (FigShare, 13 GB — no Kaggle needed), UAV-PDD2023 (Zenodo), HighRPD (Mendeley), FOD-A VOC (Google Drive).

In [ ]:
import os
for d in ['data/rdd2022', 'data/uav_pdd2023', 'data/highrpd', 'data/fod_a']:
    os.makedirs(d, exist_ok=True)

# RDD2022 — official CRDDC2022 release
!wget -c -O data/rdd2022/RDD2022_CRDDC.zip https://ndownloader.figshare.com/files/38030910
# UAV-PDD2023 — Zenodo 8429208
!wget -c -O data/uav_pdd2023/UAV-PDD2023.zip 'https://zenodo.org/api/records/8429208/files/UAV-PDD2023.zip/content'
# HighRPD — Mendeley sywswj7djj v1
!wget -c -O data/highrpd/HighRPD.zip 'https://data.mendeley.com/public-files/datasets/sywswj7djj/files/518cfb67-17f4-44a9-92a4-6d9d64cf8b83/file_downloaded'
# FOD-A — Google Drive, Pascal VOC v2.1 (412 MB)
!gdown 1RdErcq8PGRXZUOGauaACkQG44T-QyZ4x -O data/fod_a/FOD-A-VOC.zip

## 6. Extract all four

In [ ]:
extract('data/rdd2022/RDD2022_CRDDC.zip', 'data/rdd2022/extracted')
extract('data/uav_pdd2023/UAV-PDD2023.zip', 'data/uav_pdd2023/extracted')
extract('data/highrpd/HighRPD.zip', 'data/highrpd/extracted')
extract('data/fod_a/FOD-A-VOC.zip', 'data/fod_a/extracted')

## 7. ⚠️ VERIFY formats before the long run
Each source must match its loader: UAV-PDD2023 → VOC `.xml`, HighRPD → YOLO `.txt`, FOD-A → VOC `.xml`.
Check that image/label counts are non-zero, and read HighRPD's class file + a sample label to confirm the
`{0,1→crack, 2→pothole}` order. If HighRPD shows `.xml`/no `.txt`, or a different class order, **stop and report back**.

In [ ]:
from pathlib import Path
IMG = {'.jpg', '.jpeg', '.png'}
for name, d, want in [('UAV-PDD2023', 'data/uav_pdd2023/extracted', 'xml'),
                      ('HighRPD',     'data/highrpd/extracted',     'txt'),
                      ('FOD-A',       'data/fod_a/extracted',       'xml')]:
    p = Path(d)
    imgs = sum(1 for f in p.rglob('*') if f.suffix.lower() in IMG)
    xml = sum(1 for _ in p.rglob('*.xml'))
    txt = sum(1 for _ in p.rglob('*.txt'))
    flag = '' if (want == 'xml' and xml) or (want == 'txt' and txt) else '  <-- ?? expected ' + want
    print(f'{name:12s} {imgs:6d} img | {xml:6d} xml | {txt:6d} txt{flag}')

print('\n--- HighRPD: class file(s) + sample label (confirm YOLO + order) ---')
hp = Path('data/highrpd/extracted')
for f in hp.rglob('*'):
    if f.name.lower() in ('classes.txt', 'obj.names', 'data.yaml', 'classes.names') or f.suffix == '.yaml':
        print('META', f, '->', repr(f.read_text()[:300]))
samp = next(hp.rglob('*.txt'), None)
if samp:
    print('SAMPLE', samp.name, '->', repr(samp.read_text()[:200]))

print('\n--- UAV-PDD2023: sample VOC object names ---')
ux = next(Path('data/uav_pdd2023/extracted').rglob('*.xml'), None)
if ux:
    import re
    print(ux.name, '->', re.findall(r'<name>(.*?)</name>', ux.read_text()))

## 8. Merge → master split (RDD + UAV-PDD + HighRPD + FOD-A + drone)
Should print `nc: 12`. Confirm per-class counts look sane (fod should now be > 0 from FOD-A).

In [ ]:
!python src/merge_datasets.py --drone-frames frames --drone-labels drone_labels_v1 \
    --rdd-dir data/rdd2022/extracted \
    --uav-pdd-dir data/uav_pdd2023/extracted \
    --highrpd-dir data/highrpd/extracted \
    --fod-a-dir data/fod_a/extracted \
    --out datasets/master
print('\n--- data.yaml ---'); print(open('datasets/master/data.yaml').read())

from pathlib import Path
from collections import Counter
import itertools
names = ['crack','spalling','fod','faded_paint_marking','band_joint','gap_vegetation',
         'aged_surface','repair_patch','weathered_surface','surface_discoloration',
         'paint_marking','faded_surface_marking']
c = Counter()
for f in itertools.chain(Path('datasets/master/labels/train').glob('*.txt'),
                         Path('datasets/master/labels/val').glob('*.txt')):
    for ln in f.read_text().split('\n'):
        if ln.strip():
            c[int(ln.split()[0])] += 1
for i in range(12):
    print(f'{i:2d} {names[i]:22s} {c.get(i, 0)}')
print('total', sum(c.values()))

## 9. Train — YOLOv8m, 150 epochs @ 960
Uses the v1-proven knobs: `batch=16` (un-throttles AutoBatch's conservative 9) + `compile=True` (~20–40% on A100).
`project=` points at Drive so `last.pt` survives a disconnect — resume with `YOLO('.../last.pt').train(resume=True)`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from ultralytics import YOLO

YOLO('yolov8m.pt').train(
    data='datasets/master/data.yaml',
    epochs=150, imgsz=960,
    batch=16,            # 40 GB A100; use 32 on an 80 GB card
    compile=True,        # drop this line if it throws a compile/Dynamo error
    workers=8,
    close_mosaic=15, patience=50,
    project='/content/drive/MyDrive/runway_runs', name='runway_m_v2',
)

## 10. Per-class metrics on the held-out val split
Run val on the SAME Colab split (this is the trustworthy number — don't eval this model on a different machine's split).

In [ ]:
from ultralytics import YOLO
YOLO('/content/drive/MyDrive/runway_runs/runway_m_v2/weights/best.pt').val(
    data='datasets/master/data.yaml', imgsz=960, split='val')

## 11. Save artifacts to Drive (already there via project=, this just tidies + renames)

In [ ]:
import os, shutil
run = '/content/drive/MyDrive/runway_runs/runway_m_v2'
dst = '/content/drive/MyDrive/runway_m_v2'
os.makedirs(dst, exist_ok=True)
shutil.copy(f'{run}/weights/best.pt', f'{dst}/yolov8m_v2.pt')
for fn in ['results.csv', 'confusion_matrix.png', 'args.yaml']:
    if os.path.exists(f'{run}/{fn}'):
        shutil.copy(f'{run}/{fn}', dst)
print('saved ->', sorted(os.listdir(dst)))
print('best.pt MB:', round(os.path.getsize(f'{dst}/yolov8m_v2.pt') / 1e6, 1))

## 12. (Optional) Push weights + reports to GitHub
Paste a token **into this cell only** (never into chat). Revoke it afterward.

In [ ]:
TOKEN = ''  # GitHub PAT with write access; revoke after use

if TOKEN:
    import os, shutil
    run = '/content/drive/MyDrive/runway_runs/runway_m_v2'
    os.makedirs('weights', exist_ok=True)
    os.makedirs('reports/runway_m_v2', exist_ok=True)
    shutil.copy(f'{run}/weights/best.pt', 'weights/yolov8m_v2.pt')
    for fn in ['results.csv', 'confusion_matrix.png']:
        if os.path.exists(f'{run}/{fn}'):
            shutil.copy(f'{run}/{fn}', 'reports/runway_m_v2/')
    !git config user.email "shabazahmad.me1@gmail.com"
    !git config user.name "shabazahmadme1-eng"
    !git add -f weights/yolov8m_v2.pt reports/runway_m_v2
    !git -c commit.gpgsign=false commit -m "Add yolov8m v2 weights + reports (RDD+UAV-PDD+HighRPD+FOD-A, Colab A100)"
    !git remote set-url origin https://{TOKEN}@github.com/shabazahmadme1-eng/runway_screening.git
    !git push origin local/nano-bootstrap
else:
    print('No token — artifacts are in Drive from cell 11.')